In [19]:
import re
import pandas as pd
from rapidfuzz.distance import JaroWinkler
import itertools

def std_name(name):
    prefixes = r'^(?:(?:Adv|Dr|Mr|Mrs|Ms|Shri|Smt|Prof)\.?\s+)+'
    name = re.sub(prefixes, "", name, flags=re.IGNORECASE)

    lineage_field = r'\s+(?:S/o|D/o|W/o|C/o|H/o|@)\.?\s+.*$'
    name = re.sub(lineage_field, "", name, flags=re.IGNORECASE)

    name = name.split(",", 1)[0]
    name = name.split("(", 1)[0]
    name = name.replace(".", " ")

    words = name.split()
    words = [word.lower() for word in words]
    return " ".join(words)

def names_agree(a, b, threshold=0.80):
    words_a = a.split()
    words_b = b.split()
    if len(words_a) <= len(words_b):
        short, long = words_a, words_b
    else:
        short, long = words_b, words_a

    def words_match(w1, w2):
        if len(w1) == 1 or len(w2) == 1:
            return w1[0] == w2[0]
        return JaroWinkler.similarity(w1, w2) >= threshold

    for perm in itertools.permutations(long, len(short)):
        if all(words_match(s, l) for s, l in zip(short, perm)):
            return True
    return False


def match_candidates(
    election_df,
    winners_df,
    election_name_col="Candidate",
    winner_name_col="Winner Candidate",
):
    election_df = election_df.copy()
    winners_df = winners_df.copy()

    election_df["_std_name"] = election_df[election_name_col].apply(std_name)
    winners_df["_std_name"] = winners_df[winner_name_col].apply(std_name)

    matches = []

    # Outer loop: winners
    for i, winner_row in winners_df.iterrows():
        name1 = winner_row["_std_name"]

        if not name1:
            continue

        best_match = None
        best_score = 0

        # Inner loop: election candidates
        for j, election_row in election_df.iterrows():
            name2 = election_row["_std_name"]

            if not name2:
                continue

            if names_agree(name1, name2):
                score = JaroWinkler.similarity(name1, name2)

                if score > best_score:
                    best_score = score
                    best_match = {
                        "winner_candidate": winner_row[winner_name_col],
                        "winner_of_constituency": winner_row["Constituency"],
                        "election_candidate": election_row[election_name_col],
                        "similarity": score,
                        "party": winner_row.get("Winner Party", None),
                        "winner_index": i,
                        "election_index": j
                    }

        if best_match is not None:
            matches.append(best_match)

    return pd.DataFrame(matches)

In [20]:
df = pd.read_csv("loksabha_winners_2014_2019_2024_normalized.csv")

In [21]:
andhra_14_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2014)]
andhra_14 = pd.read_parquet("cleaned_data/andhra_2014.parquet")
matching_andhra_14 = match_candidates(andhra_14, andhra_14_winners)
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,146
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,234
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,15
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,69
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,185
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,282
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,209
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,183
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,87
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,309


**Manual Checking**

In [22]:
for i in list(andhra_14["Candidate"]):
    if names_agree(std_name(i), "y s avinash reddy"):
        print(i)

Avinash Reddy Yeduguri Sandinti
Shanthi Reddy
Yeluri Srilatha


In [23]:
matching_andhra_14.loc[matching_andhra_14["winner_candidate"] == "Y. S. Avinash Reddy", "election_candidate"] = "Avinash Reddy Yeduguri Sandinti"
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,146
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,234
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,15
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,69
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,185
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,282
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,209
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,183
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,87
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,309


In [24]:
andhra_19_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2019)]
andhra_19 = pd.read_parquet("cleaned_data/andhra_2019.parquet")
matching_andhra_19 = match_candidates(andhra_19, andhra_19_winners)
matching_andhra_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Goddeti Madhavi,Araku (ST),Goddeti Madhavi,1.000000,YSRCP,91,96
1,Kinjarapu Ram Mohan Naidu,Srikakulam,Rammohannaidu Kinjarapu,0.721678,TDP,92,245
2,Bellana Chandra Sekhar,Vizianagaram,Bellana Chandra Sekhar,1.000000,YSRCP,93,31
3,M. V. V. Satyanarayana,Visakhapatnam,M V V Satyanarayana,1.000000,YSRCP,94,162
4,Beesetti Venkata Satyavathi,Anakapalle,Dr Beesetti Venkata Satyavathi,1.000000,YSRCP,95,77
5,Vanga Geetha,Kakinada,Vanga Geetha Viswanatham,0.900000,YSRCP,96,295
6,Chinta Anuradha,Amalapuram (SC),Chinta Anuradha,1.000000,YSRCP,97,54
7,Margani Bharat,Rajahmundry,Margani Bharat Ram,0.955556,YSRCP,98,177
8,Raghu Rama Krishna Raju,Narasapuram,Kanumuru Raghu Rama Krishna Raju,0.761322,YSRCP,99,140
9,Kotagiri Sridhar,Eluru,K T Sreedhar,0.622685,YSRCP,100,131


In [25]:
andhra_24_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2024)]
andhra_24 = pd.read_parquet("cleaned_data/andhra_2024.parquet")
matching_andhra_24 = match_candidates(andhra_24, andhra_24_winners)
matching_andhra_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Gumma Thanuja Rani,Araku (ST),Gumma Thanuja Rani,1.000000,YSRCP,182,175
1,K. Ram Mohan Naidu,Srikakulam,Kinjarapu Rammohan Naidu,0.751103,TDP,183,222
2,Appala Naidu Kalisetti,Vizianagaram,Appalanaidu Kalisetti,0.962338,TDP,184,32
3,C. M. Ramesh,Anakapalle,C M Ramesh,1.000000,BJP,186,95
4,Tangella Uday Srinivas,Kakinada,Tangella Uday Srinivas,1.000000,Janasena Party,187,391
5,Ganti Harish Madhur,Amalapuram (SC),G M Harish,0.657018,TDP,188,149
6,Daggubati Purandeswari,Rajahmundry,Daggubati Purandheshwari,0.983333,BJP,189,115
7,Bhupathi Srinivasa Varma,Narasapuram,Bhupathi Raju Srinivasa Varma,0.907184,BJP,190,70
8,Putta Mahesh Kumar,Eluru,Putta Mahesh Kumar,1.000000,TDP,191,336
9,Vallabhaneni Balashowry,Machilipatnam,Balashowry Vallabhaneni,0.620083,Janasena Party,192,52


In [26]:
bihar_24_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2024)]
bihar_24 = pd.read_parquet("cleaned_data/bihar_2024.parquet")
matching_bihar_24 = match_candidates(bihar_24, bihar_24_winners)
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,444
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,128
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,311
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,201
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,104
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,57
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,355
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,112
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,294
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,234


**Manual Checking**

In [27]:
for i in list(bihar_24["Candidate"]):
    if names_agree(std_name(i), "lalan singh"):
        print(i)

Lal Badsah Singh
Rajiv Ranjan Singh Alias Lalan Singh


In [28]:
matching_bihar_24.loc[matching_bihar_24["winner_candidate"] == "Lalan Singh", "election_candidate"] = "Rajiv Ranjan Singh Alias Lalan Singh"
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,444
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,128
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,311
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,201
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,104
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,57
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,355
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,112
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,294
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,234


In [29]:
for i in list(bihar_24["Candidate"]):
    if names_agree(std_name(i), "abhay kushwaha"):
        print(i)

A K Singh


In [30]:
bihar_19_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2019)]
bihar_19 = pd.read_parquet("cleaned_data/bihar_2019.parquet")
matching_bihar_19 = match_candidates(bihar_19, bihar_19_winners)
matching_bihar_19


,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Baidyanath Prasad Mahto,Valmiki Nagar,Baidyanath Prasad Mahto,1.000000,JD(U),116,80
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jaiswal,1.000000,BJP,117,155
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,118,351
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,119,410
4,Sunil Kumar Pintu,Sitamarhi,Sunil Kumar,0.929412,JD(U),120,542
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,121,74
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),122,428
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),123,136
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,124,331
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,125,153


In [31]:
for i in list(bihar_19["Candidate"]):
    if names_agree(std_name(i), "ram chandra paswan"):
        print(i)

Ramchandra Paswan
Ramvilaf Paswan
Shivchandra Ram


In [32]:
matching_bihar_19.loc[matching_bihar_19["winner_candidate"] == "Ram Chandra Paswan", "election_candidate"] = "Ramchandra Paswan"
matching_bihar_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Baidyanath Prasad Mahto,Valmiki Nagar,Baidyanath Prasad Mahto,1.000000,JD(U),116,80
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jaiswal,1.000000,BJP,117,155
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,118,351
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,119,410
4,Sunil Kumar Pintu,Sitamarhi,Sunil Kumar,0.929412,JD(U),120,542
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,121,74
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),122,428
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),123,136
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,124,331
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,125,153


In [33]:
bihar_14_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2014)]
bihar_14 = pd.read_parquet("cleaned_data/bihar_2014.parquet")
matching_bihar_14 = match_candidates(bihar_14, bihar_14_winners)
matching_bihar_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Satish Chandra Dubey,Valmiki Nagar,Satish Dubey,0.920000,BJP,25,485
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jayaswal,0.943663,BJP,26,134
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,27,381
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,28,430
4,Ram Kumar Sharma,Sitamarhi,Ram Kumar Sharma,1.000000,Rashtriya Lok Samta Party,29,423
5,Hukmdev Narayan Yadav,Madhubani,Hukum Deo Narayan Yadav,0.900121,BJP,30,165
6,Birendra Chaudhary,Jhanjharpur,Bijendra Chaudhary,0.938998,BJP,31,75
7,Ranjeet Ranjan,Supaul,Ranjeet Ranjan,1.000000,INC,32,450
8,Taslimuddin,Araria,Taslim Uddin,0.983333,RJD,33,565
9,Tariq Anwar,Katihar,Md Tariq Anwar,0.837662,NCP,35,270


In [34]:
gujarat_14_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2014)]
gujarat_14 = pd.read_parquet("cleaned_data/gujarat_2014.parquet")
matching_gujarat_14 = match_candidates(gujarat_14, gujarat_14_winners)
matching_gujarat_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhmashi,0.694264,BJP,65,43
1,Haribhai Chaudhary,Banaskantha,Harilal Ranchodbhai Chauhan,0.844227,BJP,66,81
2,Liladhar Vaghela,Patan,Vaghela Liladharbhai Khodaji,0.711310,BJP,67,313
3,Jayshreeben Patel,Mahesana,Patel Jayshreeben Kanubhai,0.747360,BJP,68,193
4,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,69,235
5,L. K. Advani,Gandhinagar,Lal Krishna Advani,0.746667,BJP,70,117
6,Paresh Rawal,Ahmedabad East,Paresh Dahyalal Rawal,0.880952,BJP,71,172
7,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai Premjibhai,0.667582,BJP,72,66
8,Devajibhai Fatepara,Surendranagar,Fatepara Devajibhai Govindbhai,0.719883,BJP,73,68
9,Mohan Kundariya,Rajkot,Mohanbhai Kalyanjibhai Kundariya,0.641667,BJP,74,148


In [35]:
gujarat_19_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2019)]
gujarat_19 = pd.read_parquet("cleaned_data/gujarat_2019.parquet")
matching_gujarat_19 = match_candidates(gujarat_19, gujarat_19_winners)
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [36]:
for i in list(gujarat_19["Candidate"]):
    if names_agree(std_name(i), std_name("Rajesh Chudasama")):
        print(i)

Chudasama Rajeshbhai Naranbhai
Rakeshbhai


In [37]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "Rajesh Chudasama", "election_candidate"] = "Chudasama Rajeshbhai Naranbhai"
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [38]:
for i in list(gujarat_19["Candidate"]):
    if names_agree(std_name(i), std_name("C. R. Patil")):
        print(i)

Chandansinh
Chandraben
Patil Chandrakant
Rakeshbhai
Reshma Patel
Rita Captain


In [39]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "C. R. Patil", "election_candidate"] = "Patil Chandrakant"
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [40]:
gujarat_24_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2024)]
gujarat_24 = pd.read_parquet("cleaned_data/gujarat_2024.parquet")
matching_gujarat_24 = match_candidates(gujarat_24, gujarat_24_winners)
matching_gujarat_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,247,58
1,Geni Thakor,Banaskantha,Geniben Nagaji Thakor,0.610390,INC,248,88
2,Bharatsinhji Dabhi,Patan,Dabhi Bharatsinhji Shankarji,0.732804,BJP,249,63
3,Haribhai Patel,Mahesana,Haribhai Patel,1.000000,BJP,250,94
4,Shobhanaben Baraiya,Sabarkantha,Shobhanaben Mahendrasinh Baraiya,0.887171,BJP,251,229
5,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,252,8
6,Hasmukh Patel,Ahmedabad East,Hasmukhbhai Patel,0.937557,BJP,253,97
7,Dinesh Makwana,Ahmedabad West,Dineshbhai Makwana,0.898413,BJP,254,79
8,Chandubhai Shihora,Surendranagar,Chandubhai Chhaganbhai Shihora,0.886667,BJP,255,50
9,Parshottam Rupala,Rajkot,Parshottambhai Rupala,0.926611,BJP,256,184


**Adding new columns to data**

In [41]:
def process_election_data(state, year, matching_df):
    file_path = f"cleaned_data/{state}_{year}.parquet"
    df = pd.read_parquet(file_path)
    
    df["winner"] = False
    df["constituency"] = ""
    
    mapping_df = matching_df[['election_candidate', 'winner_of_constituency']].copy()
    mapping_df.columns = ['Candidate', 'matched_constituency']
    
    df = df.merge(mapping_df, on='Candidate', how='left')
    
    is_winner = df['matched_constituency'].notna()
    df.loc[is_winner, 'winner'] = True
    df.loc[is_winner, 'constituency'] = df.loc[is_winner, 'matched_constituency']
    
    df.drop(columns=['matched_constituency'], inplace=True)
    return df

In [43]:
datasets = [
    ("andhra", 2014, matching_andhra_14),
    ("andhra", 2019, matching_andhra_19),
    ("andhra", 2024, matching_andhra_24),
    ("bihar", 2014, matching_bihar_14),
    ("bihar", 2019, matching_bihar_19),
    ("bihar", 2024, matching_bihar_24),
    ("gujarat", 2014, matching_gujarat_14),
    ("gujarat", 2019, matching_gujarat_19),
    ("gujarat", 2024, matching_gujarat_24),
]